# R pathview versus Python Pathview Plus

I used this notebook for one specific question:

> If R and Python receive the same pathway, genes, numbers, colors, and condition order, do they both make the requested left/right half-and-half node?

Yes. The controlled native-PNG comparison passed.

In [ ]:
from pathlib import Path
import json
import os
import sys

HERE = Path.cwd().resolve()
SEARCH_FOLDERS = (HERE, *HERE.parents)
PROJECT_CANDIDATES = (
    *SEARCH_FOLDERS,
    *(folder / "pygage-pathview-validation" for folder in SEARCH_FOLDERS),
)
ROOT = next(
    (
        folder for folder in PROJECT_CANDIDATES
        if (folder / ".venv").exists() and (folder / "scripts").exists()
    ),
    HERE,
)

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".mplconfig"))
print("Validation folder:", ROOT)

## Part 1 — Execute both implementations

R pathview and Python Pathview Plus use the same frozen `hsa04110.xml` and `hsa04110.png` from Bioconductor pathview 1.52.0.

In [ ]:
import subprocess
import polars as pl
from IPython.display import Image, display

env = os.environ.copy()
env["R_LIBS_USER"] = str(ROOT / ".r-library")
subprocess.run(
    ["Rscript", str(ROOT / "scripts" / "run_r_pathview.R")],
    cwd=ROOT,
    env=env,
    check=True,
)
subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "run_pathview_validation.py")],
    cwd=ROOT,
    env=env,
    check=True,
)
subprocess.run(
    [sys.executable, str(ROOT / "scripts" / "compare_r_python.py")],
    cwd=ROOT,
    env=env,
    check=True,
)

In [ ]:
r_report = json.loads((ROOT / "results" / "pathview_r" / "validation.json").read_text())
py_report = json.loads((ROOT / "results" / "pathview_python" / "validation.json").read_text())
pl.DataFrame([
    {
        "implementation": "R pathview",
        "version": r_report["version"],
        "environment": r_report["r_version"],
    },
    {
        "implementation": "Python Pathview Plus",
        "version": py_report["distribution_version"],
        "environment": "Python " + py_report["python"],
    },
])

## Part 2 — Exact controlled input

I kept the input columns in this order:

| Position | Condition | Value for CDKN2A | Expected piece |
|---:|---|---:|---|
| 1 | Classical | -2 | left, green |
| 2 | Basal | +2 | right, red |

Explicit hex colors are important. R's named `green` and Matplotlib's named `green` are different shades even when both programs behave correctly.

In [ ]:
comparison = pl.read_csv(
    ROOT / "results" / "comparison" / "mapped_node_comparison.csv"
)
comparison

The absolute differences are zero. Both tools mapped the same condition values to the same KEGG coordinates.

In [ ]:
display(Image(
    filename=str(ROOT / "results" / "comparison" / "r_vs_python_half_half.png"),
    width=1200,
))

## Part 3 — Pixel-direction proof

Whole-image pixel identity is not required because the renderers handle PNG colors and output cropping differently. I checked the same CDKN2A coordinates and counted where green and red pixels were concentrated.

In [ ]:
comparison_json = json.loads(
    (ROOT / "results" / "comparison" / "comparison.json").read_text()
)
orientation = comparison_json["pixel_orientation"]
pl.DataFrame([
    {"implementation": "Python", **orientation["python_pixel_counts"]},
    {"implementation": "R", **orientation["r_pixel_counts"]},
])

In both rows, left-green dominates right-green and right-red dominates left-red. That proves the same column order visually.

## Part 4 — What is the same and what is different?

| Area | R pathview 1.52.0 | Python Pathview Plus 2.0.2 |
|---|---|---|
| One-state native PNG | Yes | Yes |
| Two-state native PNG | Left/right bands | Left/right bands |
| Three-state native PNG | Ordered bands | Ordered bands |
| Graph/PDF multi-state | Yes | Uses first state only |
| SVG output | Not a main pathview output | Yes, node-based SVG |
| SBGN/non-KEGG integration | R pathview is KEGG-focused | Separate utilities exist; not yet end-to-end through `pathview()` |

`split.group` in R is **not** the half-and-half feature. The half-and-half feature is `multi.state = TRUE` with two value columns.

## Final verdict

For the requested basic classical example, use one value column. For the requested half-and-half view, use two columns in the order you want displayed.

The controlled comparison passed at three levels:

1. same mapped gene-node coordinates;
2. identical state values at those coordinates;
3. correct left-green/right-red pixel dominance in both R and Python.